# 5.8 Regression Metrics

**Machine Learning Mastery: Mathematics to MLOps** · Chapter 5: Linear Regression

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RithikRaj129/ml-mastery/blob/main/chapter-05-linear-regression/05.08_regression-metrics.ipynb)

Companion code for Section 5.8 of the book. Run the cells in order from top to bottom.

**Example 5.8.1**

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import (mean_absolute_error, mean_squared_error,
                              mean_absolute_percentage_error, r2_score)

def all_regression_metrics(y_true, y_pred, n_features=None, name='Model'):
    '''Compute and display all regression metrics.'''
    n    = len(y_true)
    mae  = mean_absolute_error(y_true, y_pred)
    mse  = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    mape = mean_absolute_percentage_error(y_true, y_pred) * 100  # %
    r2   = r2_score(y_true, y_pred)

    # Adjusted R²
    if n_features is not None:
        r2_adj = 1 - (1 - r2) * (n - 1) / (n - n_features - 1)
    else:
        r2_adj = None

    # Median Absolute Error - robust central tendency of errors
    medae = np.median(np.abs(y_true - y_pred))

    # Max Error - worst case
    max_err = np.max(np.abs(y_true - y_pred))

    print(f'\n=== {name} ===')
    print(f'  MAE           = {mae:.4f}  (avg absolute error)')
    print(f'  MSE           = {mse:.4f}  (avg squared error)')
    print(f'  RMSE          = {rmse:.4f}  (root mean squared error)')
    print(f'  MAPE          = {mape:.2f}%  (avg percentage error)')
    print(f'  Median AE     = {medae:.4f}')
    print(f'  Max Error     = {max_err:.4f}')
    print(f'  R²            = {r2:.6f}  (variance explained)')
    if r2_adj is not None:
        print(f'  Adjusted R²   = {r2_adj:.6f}  (R² penalised for {n_features} features)')

    return {'MAE':mae,'MSE':mse,'RMSE':rmse,'MAPE':mape,
            'MedAE':medae,'MaxErr':max_err,'R2':r2,'R2_adj':r2_adj}

# ── Illustrate what each metric captures ─────────────────────────
np.random.seed(42)
n = 100
y_true   = np.random.normal(100, 20, n)
y_good   = y_true + np.random.normal(0, 5, n)   # Good model
y_biased = y_true + 15                           # Systematic overestimation
y_outlier= y_true + np.random.normal(0, 3, n)   # Mostly good, few huge errors
y_outlier[np.random.choice(n, 5)] += np.random.uniform(50, 100, 5)  # 5 outliers

metrics_good    = all_regression_metrics(y_true, y_good,    name='Good Model')
metrics_biased  = all_regression_metrics(y_true, y_biased,  name='Biased Model (+15 offset)')
metrics_outlier = all_regression_metrics(y_true, y_outlier, name='Outlier Sensitive')

# ── Metric sensitivity comparison ────────────────────────────────
print('\n=== KEY INSIGHT: Different Metrics Reveal Different Problems ===')
comparison = pd.DataFrame({
    'Metric':  ['MAE','RMSE','MAPE%','R²'],
    'Good':    [metrics_good['MAE'], metrics_good['RMSE'],
                metrics_good['MAPE'], metrics_good['R2']],
    'Biased':  [metrics_biased['MAE'], metrics_biased['RMSE'],
                metrics_biased['MAPE'], metrics_biased['R2']],
    'Outliers':[metrics_outlier['MAE'], metrics_outlier['RMSE'],
                metrics_outlier['MAPE'], metrics_outlier['R2']],
})
print(comparison.round(4).to_string(index=False))

# ── Visualise error distributions ────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('Error Distributions for Three Models', fontsize=13, fontweight='bold')

for ax, y_pred, title, color in zip(
    axes,
    [y_good, y_biased, y_outlier],
    ['Good Model', 'Biased Model', 'Outlier-Prone Model'],
    ['#1976D2', '#E53935', '#FF6F00']
):
    errors = y_true - y_pred
    ax.hist(errors, bins=30, color=color, alpha=0.7, edgecolor='white')
    ax.axvline(0, color='black', lw=2, linestyle='--')
    ax.axvline(errors.mean(), color='red', lw=2, label=f'Mean={errors.mean():.1f}')
    mae_val  = np.mean(np.abs(errors))
    rmse_val = np.sqrt(np.mean(errors**2))
    ax.set_title(f'{title}\nMAE={mae_val:.1f}  RMSE={rmse_val:.1f}', fontsize=11)
    ax.set_xlabel('Error (y - ŷ)'); ax.set_ylabel('Frequency')
    ax.legend(); ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('regression_metrics_comparison.png', dpi=150, bbox_inches='tight')
plt.show()